# Week 0 · Part A: Train your first language model

**The LLM Resident** · the lab for *The Map of the Territory*

Thirty minutes, no GPU needed, no ML background assumed. You will train the smallest model that is honestly called a language model: it learns which letter tends to follow which. Then it samples name-shaped strings—some new, some repeated—and you'll measure it on spellings it never saw during training.

Run every cell top to bottom (`Shift+Enter`). The `assert` lines are self-checks: if a cell runs silently, its basic invariants hold.

## 1 · Get the data

The file contains 32,033 name records derived from 2018 US Social Security Administration data and distributed with Andrej Karpathy's [*makemore*](https://github.com/karpathy/makemore/tree/988aa59e4d8fefa526d06f3b453ad116258398d4). We pin the exact source commit so the data cannot silently change underneath the lab.

In [ ]:
import hashlib, os, urllib.request

DATA_COMMIT = "988aa59e4d8fefa526d06f3b453ad116258398d4"
DATA_URL = f"https://raw.githubusercontent.com/karpathy/makemore/{DATA_COMMIT}/names.txt"
DATA_PATH = "names-988aa59.txt"
EXPECTED_SHA256 = "0a30b5557f192f32ab962680889aac5f6fda0f4cecf40a6d0b5694f58ea8cc4d"

if not os.path.exists(DATA_PATH):
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

with open(DATA_PATH, "rb") as f:
    data = f.read()

assert hashlib.sha256(data).hexdigest() == EXPECTED_SHA256, "dataset checksum changed"
print("dataset ready:", len(data), "bytes · source commit:", DATA_COMMIT[:8])

## 2 · Look at it

Always look at your data before you model it. This file contains duplicate spellings, so we will make that visible and remove duplicates before splitting. Otherwise the same spelling could land in both training and evaluation.

In [ ]:
words = data.decode("utf-8").splitlines()
unique_words = sorted(set(words))

print("records:", len(words))
print("unique spellings:", len(unique_words))
print("first five:", words[:5])
print("shortest:", min(words, key=len), "· longest:", max(words, key=len))

assert len(words) == 32033, "expected 32,033 names; re-run the download cell"
assert len(unique_words) == 29494, "expected 29,494 unique spellings"
assert all(w.isalpha() and w.islower() for w in words), "unexpected characters in the data"

## 3 · Split before you learn

We shuffle the unique spellings once, deterministically, then reserve 10% for validation and 10% for the final test. The model may count only the training split. Validation helps us compare ideas; the test split stays untouched until the end.

In [ ]:
import random

split_words = unique_words.copy()
random.Random(42).shuffle(split_words)
n_train = int(0.80 * len(split_words))
n_val = int(0.10 * len(split_words))

train_words = split_words[:n_train]
val_words = split_words[n_train:n_train + n_val]
test_words = split_words[n_train + n_val:]

print("train:", len(train_words), "· validation:", len(val_words), "· test:", len(test_words))
assert (len(train_words), len(val_words), len(test_words)) == (23595, 2949, 2950)
assert set(train_words).isdisjoint(val_words)
assert set(train_words).isdisjoint(test_words)
assert set(val_words).isdisjoint(test_words)

## 4 · Build a tokenizer (a tiny one)

Models eat numbers, not letters. Our vocabulary is 27 tokens: the letters `a–z`, plus `.` which marks both the **start** and the **end** of a name. Week 2 replaces this with a real BPE tokenizer you'll build from scratch; the idea is identical.

In [ ]:
chars = sorted(set("".join(train_words)))     # fit the vocabulary on training data only
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0                                 # '.' = start-of-name AND end-of-name
itos = {i: s for s, i in stoi.items()}

print("vocabulary size:", len(stoi))
assert len(stoi) == 27 and stoi["."] == 0 and itos[1] == "a"

## 5 · Train the model (yes, really)

The entire model is a 27×27 table: `N[a, b]` counts how often character `b` followed character `a` in the **training split only**. This table estimates `P(next character | current character)`. Neural causal language models estimate the analogous next-token distribution from learned parameters and a much longer context.

In [ ]:
import torch

N = torch.zeros((27, 27), dtype=torch.int32)   # [V, V]; row = current char, col = next char
for w in train_words:
    cs = ["."] + list(w) + ["."]
    for c1, c2 in zip(cs, cs[1:]):
        N[stoi[c1], stoi[c2]] += 1

assert N.shape == (27, 27)
assert int(N.sum()) == sum(len(w) + 1 for w in train_words), "every training name contributes len+1 transitions"

r, c = divmod(int(N.argmax()), 27)
print("total transitions counted:", int(N.sum()))
print(f"most common transition: {itos[r]!r} -> {itos[c]!r} ({int(N[r, c])} times)")

## 6 · Counts → probabilities

Each row becomes a probability distribution over the next character. The `+1` is **add-one smoothing**: an unseen transition receives a small probability instead of zero. That matters now because validation and test contain spellings the model was not allowed to count.

In [ ]:
P = (N + 1).float()          # smoothing: no zero probabilities
P /= P.sum(1, keepdim=True)  # every row now sums to 1.0

assert torch.allclose(P.sum(1), torch.ones(27)), "rows must sum to 1"
print("P[stoi['q']]: what tends to follow 'q'? top guess:", itos[int(P[stoi['q']].argmax())])

## 7 · Generate

This is the basic autoregressive generation loop used by GPT-style causal language models: look at the distribution for the current context, sample one token, append it, and repeat. The model producing that distribution is a counting table here and a transformer there.

The seed makes this cell repeatable in the same environment, but [PyTorch does not guarantee identical random results across releases or platforms](https://docs.pytorch.org/docs/stable/notes/randomness.html). We print the version alongside the samples; the outputs reported for this lab were verified on PyTorch 2.13.0 on CPU.

In [ ]:
g = torch.Generator(device="cpu").manual_seed(42)
train_lookup = set(train_words)

print("PyTorch:", torch.__version__, "· device: CPU")
for _ in range(10):
    out, ix = [], 0
    while True:
        ix = int(torch.multinomial(P[ix], num_samples=1, generator=g))
        if ix == 0:
            break
        out.append(itos[ix])
    name = "".join(out)
    status = "seen in training" if name in train_lookup else "not in training"
    print(f"{name or '<empty>'}  [{status}]")

## 8 · Measure it without peeking

Some samples look plausible; some are garbage. "Looks OK to me" is not evaluation. We ask how surprised the model is, on average, by each observed character transition. The metric is **average negative log-likelihood (NLL)**, measured here in natural-log units, or **nats per transition**. Lower is better.

A model that guesses uniformly among all 27 tokens scores `ln(27) ≈ 3.296`. Training NLL tells us how well the table fits what it counted. Validation and test NLL tell us how well it predicts held-out spellings.

In [ ]:
import math

def average_nll(dataset):
    total_nll, transitions = 0.0, 0
    for w in dataset:
        cs = ["."] + list(w) + ["."]
        for c1, c2 in zip(cs, cs[1:]):
            total_nll -= float(torch.log(P[stoi[c1], stoi[c2]]))
            transitions += 1
    return total_nll / transitions, transitions

uniform_nll = math.log(len(stoi))
scores = {}
print(f"uniform baseline : {uniform_nll:.4f} nats/transition")
for label, dataset in [("train", train_words), ("validation", val_words), ("test", test_words)]:
    score, transitions = average_nll(dataset)
    scores[label] = score
    print(f"{label:16s}: {score:.4f} nats/transition  ({transitions:,} transitions)")

assert all(math.isfinite(score) for score in scores.values())
assert scores["validation"] < uniform_nll and scores["test"] < uniform_nll
print()
print("The held-out scores beat uniform guessing: the model learned signal that transfers.")

## What you just did

- **Trained a causal language model.** It estimated a next-character distribution from the training split and sampled from it. A sample can be new to training or repeat something the model saw; sampling does not guarantee novelty.
- **Evaluated without training on the answers.** Validation and test spellings were excluded before counting. Their NLLs beat uniform guessing, so the model learned real signal that transfers.
- **Found the limit.** One character of context can produce locally plausible transitions while losing the shape of the whole name. The samples make that failure visible; the held-out score makes comparison possible.
- **Set up the residency's core loop:** data → split → model → generate → evaluate. Every week deepens one of those arrows.

**Next:** Week 0 · Part B builds your toolchain and your shape-fluency. In Week 1, validation guides model choices; after those choices are frozen, the test split gets one final comparison against this baseline.

*The LLM Residency: six months, ten hours a week, no magic.*